# Synthetic Aperture Radar (SAR) Data Metrics Calculations

TAT-C is used for orbit propagation (satellite states), OrbitPy is used for access interval calculations, while InstruPy is used for the data metrics calculations.

In [ ]:
from joblib import Parallel, delayed

import warnings
import tempfile
import os, shutil
import csv


import json
from datetime import datetime, timedelta, timezone
from shapely.geometry import box, mapping
from scipy.stats import hmean
import pandas as pd

from astropy.time import Time as AstroPy_Time

from scipy.spatial.transform import Rotation as Scipy_Rotation

from pydantic import AwareDatetime

from tatc.schemas import (
    Instrument as TATC_Instrument,
    Satellite as TATC_Satellite,
    TwoLineElements,
    Point,
)
from tatc.analysis import collect_orbit_track, OrbitCoordinate, OrbitOutput
from tatc.analysis import (
    collect_multi_observations,
    aggregate_observations,
    reduce_observations,
)

from orbitpy.util import (
    OrbitState as OrbitPy_OrbitState,
    Spacecraft as OrbitPy_Spacecraft,
    SpacecraftBus as OrbitPy_SpacecraftBus,
)
from orbitpy.propagator import (
    J2AnalyticalPropagator as OrbitPy_J2AnalyticalPropagator,
    SGP4Propagator as OrbitPy_SGP4Propagator,
)
from orbitpy.coveragecalculator import (
    GridCoverage as OrbitPy_GridCoverage,
    find_access_intervals as OrbitPy_find_access_intervals,
)
from orbitpy.grid import Grid as OrbitPy_Grid

from instrupy import Instrument as InstruPy_Instrument

\
from eose.propagation import (
    PropagationSample,
    PropagationRecord,
    PropagationRequest,
    PropagationResponse,
)
from eose.orbits import GeneralPerturbationsOrbitState, Propagator
from eose.satellites import Satellite
from eose.utils import (
    CartesianReferenceFrame,
    PlanetaryCoordinateReferenceSystem,
    Quaternion,
    FixedOrientation,
)

from eose.access import (
    AccessSample,
    AccessRecord,
    AccessRequest,
    AccessResponse,
)
from eose.grids import UniformAngularGrid

from instrupy.synthetic_aperture_radar_model import SyntheticApertureRadarModel as InstruPy_SyntheticApertureRadarModel

from eose.instruments import RectangularGeometry, Antenna, SinglePolStripMapSAR
from eose.datametrics import (
    DataMetricsRequest,
    SinglePolStripMapSARInstantaneous,
    DataMetricsSample,
    DataMetricsRecord,
    DataMetricsResponse,
)

pd.set_option("display.max_rows", None)

## Define Mission Parameters

In [ ]:
# define the orbit and the instrument
# Note that the ISS orbit has been used, and not the Seasat orbit.
iss_omm_str = '[{"OBJECT_NAME":"ISS (ZARYA)","OBJECT_ID":"1998-067A","EPOCH":"2024-06-07T09:53:34.728000","MEAN_MOTION":15.50975122,"ECCENTRICITY":0.0005669,"INCLINATION":51.6419,"RA_OF_ASC_NODE":3.7199,"ARG_OF_PERICENTER":284.672,"MEAN_ANOMALY":139.0837,"EPHEMERIS_TYPE":0,"CLASSIFICATION_TYPE":"U","NORAD_CAT_ID":25544,"ELEMENT_SET_NO":999,"REV_AT_EPOCH":45703,"BSTAR":0.00033759,"MEAN_MOTION_DOT":0.00019541,"MEAN_MOTION_DDOT":0}]'

iss_omm = json.loads(iss_omm_str)[0]

seasat_sar = SinglePolStripMapSAR(
                id="Seasat-SAR",
                orientation= list([0, 0.1779435, 0, 0.9840407]), # +20.5 deg roll about x-axis (roll),
                field_of_view= RectangularGeometry(angle_height= 1.107372347119861, angle_width= 5.485594497306811),
                scene_field_of_view= RectangularGeometry(angle_height= 30, angle_width= 5.485594497306811),
                bits_per_pixel= 5,
                pulse_width= 33.4e-6,
                antenna= Antenna(shape=Antenna.RectangularAntennaShape(height=10.7, width=2.16), aperture_excitation_profile='UNIFORM', aperture_efficiency=0.6),
                operating_frequency= 1.2757e9,
                peak_transmit_power= 1000,
                chirp_bandwidth= 19e6,
                minimum_prf= 1463,
                maximum_prf= 1686,
                scene_noise_temp= 290,
                system_noise_figure= 5.11,
                radar_loss= 3.5,
            )
satellites = [
    Satellite(
        id="Seasat",
        orbit=GeneralPerturbationsOrbitState.from_omm(iss_omm),
        payloads=[seasat_sar],
    )
]

targets = UniformAngularGrid(
    delta_latitude=0.1, delta_longitude=0.1, region=mapping(box(20, -10, 30, 10))
).as_targets()

mission_start = datetime(2024, 1, 1, tzinfo=timezone.utc)
mission_duration = timedelta(hours=24)
propagate_time_step = timedelta(minutes=0.5)

## Run Propagation with TAT-C

To get satellite states

In [ ]:
def propagate_tatc(request: PropagationRequest) -> PropagationResponse:
    if request.propagator != Propagator.SGP4:
        raise RuntimeError("TAT-C only supports SGP4 propagator.")
    orbit_tracks = Parallel(-1)(
        delayed(collect_orbit_track)(
            TATC_Satellite(
                name=satellite.id,
                orbit=TwoLineElements(tle=satellite.orbit.to_tle()),
            ),
            TATC_Instrument(name="Instrument"),  # dummy entry
            pd.date_range(
                request.start,
                request.start + request.duration,
                freq=request.time_step,
            ),
            coordinates=(
                OrbitCoordinate.ECI
                if request.frame == CartesianReferenceFrame.ICRF
                else OrbitCoordinate.ECEF
            ),
            orbit_output=OrbitOutput.POSITION_VELOCITY,
        )
        for satellite in request.satellites
    )
    return PropagationResponse(
        **request.model_dump(exclude="satellite_records"),
        satellite_records=[
            PropagationRecord(
                satellite_id=satellite.id,
                samples=orbit_tracks[i].apply(
                    lambda r: PropagationSample(
                        time=r.time,
                        frame=request.frame,
                        position=r.geometry.coords[0],
                        velocity=r.velocity.coords[0],
                    ),
                    axis=1,
                ),
            )
            for i, satellite in enumerate(request.satellites)
        ],
    )


propagate_request = PropagationRequest(
    satellites=satellites,
    start=mission_start,
    duration=mission_duration,
    time_step=propagate_time_step,
    frame=CartesianReferenceFrame.ICRF,
    propagator=Propagator.SGP4,
)

#display(propagate_request.model_dump_json())

tatc_propagation_response = propagate_tatc(propagate_request)

#display(propagation_response.model_dump_json())

propagation_data = tatc_propagation_response.as_dataframe()

display(propagation_data)

## Run OrbitPy Access Calculator

Run access calculations with OrbitPy. The orbit propagator utiized should match with the one used in propogation calculation.

In [ ]:
def access_orbitpy(request: AccessRequest) -> AccessResponse:

    # create a temporary directory to hold temporary files
    script_directory = os.path.dirname(os.path.abspath("__file__"))
    temp_dir = os.path.join(script_directory, "temp")
    os.makedirs(temp_dir, exist_ok=True)

    #### Enumerate and convert from EOSE-API satellites to OrbitPy satellite objects. ####
    # (Enumeration generates distinct orbit-instrument pairs for satellites equipped with multiple instruments.)
    OrbitPy_Satellites = []
    for satellite in request.satellites:
        for instru in satellite.payloads:
            if instru.id in request.payload_ids:
                instru_type = instru.type
                if instru_type == "BasicSensor" or "SinglePolStripMapSAR":

                    if instru.field_of_view.type == "CircularGeometry":
                        instupy_fov_geom = {
                            "shape": "CIRCULAR",
                            "diameter": instru.field_of_view.diameter,
                        }
                    elif instru.field_of_view.type == "RectangularGeometry":
                        instupy_fov_geom = {
                            "shape": "RECTANGULAR",
                            "angleHeight": instru.field_of_view.angle_height,
                            "angleWidth": instru.field_of_view.angle_width,
                        }
                    else:
                        raise ValueError(
                            f"Only Circular and Rectangular geometries are supported and not {instru.field_of_view.type}"
                        )

                    # Convert orientation in Quaternion to Euler rotations
                    r = Scipy_Rotation.from_quat(list(instru.orientation))
                    (x, y, z) = r.as_euler(
                        "XYZ", degrees=True
                    )  # Conventions 'XYZ' are for intrinsic rotations (used by OrbitPy), while 'xyz' are for extrinsic rotations.

                    # A Basic Sensor type is initialized regardless of the actual sensor type, as OrbitPy only requires the FOV, sceneFOV, and orientation information for access calculations.
                    instrupy_sensor = InstruPy_Instrument.from_dict(
                        {
                            "@type": "Basic Sensor",
                            "orientation": {
                                "referenceFrame": "SC_BODY_FIXED",
                                "convention": "REF_FRAME_ALIGNED",
                            },
                            "fieldOfViewGeometry": instupy_fov_geom,
                            "orientation": {
                                "referenceFrame": "NADIR_POINTING",
                                "convention": "XYZ",
                                "xRotation": x,
                                "yRotation": y,
                                "zRotation": z,
                            },
                            "@id": instru.id,
                        }
                    )
                else:
                    raise ValueError(
                        f"{instru_type} instrument type is not supported in this script. Only 'SinglePolStripMapSAR' or 'BasicSensor' instrument type is supported."
                    )

                tle = satellite.orbit.to_tle()

                orbit_state = OrbitPy_OrbitState.from_dict(
                    {
                        "tle": {
                            "tle_line0": "Unknown",
                            "tle_line1": tle[0],
                            "tle_line2": tle[1],
                        }
                    }
                )
                if satellite.satellite_bus is None or satellite.satellite_bus.orientation == FixedOrientation.NADIR_GEOCENTRIC:
                    orbitpy_sat_bus = OrbitPy_SpacecraftBus.from_dict({
                        "orientation": {
                            "referenceFrame": "Nadir_pointing",
                            "convention": "REF_FRAME_ALIGNED",
                        }
                    })
                else:
                    warnings.warn(
                        "OrbitPy only processes spacecraft-bus orientation aligned with the NADIR_GEOCENTRIC frame. "
                        "To account for off-nadir instrument viewing, please specify the instrument orientation relative "
                        "to the NADIR_GEOCENTRIC frame.",
                        UserWarning,
                    )


                sat = OrbitPy_Spacecraft(
                    _id=satellite.id,
                    orbitState=orbit_state,
                    spacecraftBus=orbitpy_sat_bus,
                    instrument=[instrupy_sensor],
                )

                OrbitPy_Satellites.append(sat)

    #### Format the Target points into OrbitPy Grid object ####
    lon = []
    lat = []
    target_id = []
    # iterate through the Target points
    for tp in request.targets:
        if tp.crs == PlanetaryCoordinateReferenceSystem.EPSG_4326 or tp.crs is None:
            lon.append(tp.position[0])
            lat.append(tp.position[1])
            target_id.append(tp.id)
        else:
            raise ValueError(
                f"{tp.crs} CRS is not supported by OrbitPy. Only 'EPSG_4326' CRS is supported."
            )

    row_to_target_id = (
        {}
    )  # Dictionary to map row numbers ('GP index' in OrbitPy) to target_id
    orbitpy_custom_grid = None
    with tempfile.NamedTemporaryFile(
        mode="w+t", delete=False, dir=temp_dir
    ) as grid_file:
        writer = csv.writer(grid_file)
        writer.writerow(["lat [deg]", "lon [deg]", "id"])

        for row_num, (lat_val, lon_val, target_id_val) in enumerate(
            zip(lat, lon, target_id)
        ):
            writer.writerow([lat_val, lon_val, target_id_val])
            row_to_target_id[row_num] = target_id_val

    orbitpy_custom_grid = OrbitPy_Grid.from_customgrid_dict(
        {"@type": "customGrid", "covGridFilePath": grid_file.name}
    )

    #### run propagation and coverage with OrbitPy ####
    step_size_s = request.time_step.total_seconds()
    if request.propagator != Propagator.J2:
        propagator = OrbitPy_J2AnalyticalPropagator.from_dict(
            {"@type": "J2 ANALYTICAL PROPAGATOR", "stepSize": step_size_s}
        )
    elif request.propagator != Propagator.SGP4:
        propagator = OrbitPy_SGP4Propagator.from_dict(
            {"@type": "SGP4 PROPAGATOR", "stepSize": step_size_s}
        )
    else:
        raise RuntimeError("OrbitPy only supports J2 and SGP4 propagators.")

    #### Convert request time to Julian Date UT1####
    utc_dt = request.start.astimezone(
        timezone.utc
    )  # Convert to UTC (if not already in UTC)
    astropy_utc_time = AstroPy_Time(
        utc_dt, scale="utc"
    )  # Convert to astropy Time object
    astropy_ut1_time = astropy_utc_time.ut1  # Convert to UT1 scale

    start_date_dict = {"@type": "JULIAN_DATE_UT1", "jd": astropy_ut1_time.jd}
    start_date = OrbitPy_OrbitState.date_from_dict(
        start_date_dict
    )  # assumed that the time scale is UT1.
    duration = request.duration.total_seconds() / 86400.0

    for orbitpy_sat in OrbitPy_Satellites:

        # run propagation with OrbitPy
        with tempfile.NamedTemporaryFile(
            mode="w+t", delete=False, dir=temp_dir
        ) as state_cart_file:  # store satellite states in a temporary file.
            propagator.execute(
                orbitpy_sat, start_date, state_cart_file.name, None, duration
            )

            # run access calculations with OrbitPy
            with tempfile.NamedTemporaryFile(
                mode="w+t", delete=False, dir=temp_dir
            ) as access_fl:
                cov_calc = OrbitPy_GridCoverage(
                    grid=orbitpy_custom_grid,
                    spacecraft=orbitpy_sat,
                    state_cart_file=state_cart_file.name,
                )
                instru_id = orbitpy_sat.get_instrument().get_id()
                cov_calc.execute(
                    instru_id=instru_id,
                    mode_id=None,
                    use_field_of_regard=True,
                    out_file_access=access_fl.name,
                    mid_access_only=False,
                )
                intervals_df = OrbitPy_find_access_intervals(access_fl.name)

                grouped_intervals = intervals_df.groupby("GP index")
                access_records = []  # record of accesses at each target point
                for gp_index, group in grouped_intervals:
                    # Iterate over each record in the group
                    access_sample = []
                    for index, row in group.iterrows():
                        access_start = request.start + timedelta(
                            seconds=row["Start time index"] * step_size_s
                        )
                        access_duration = timedelta(
                            seconds=row["Duration"] * step_size_s
                        )
                        # form access sample
                        access_sample.append(
                            AccessSample(
                                satellite_id=orbitpy_sat._id,
                                instrument_id=instru_id,
                                start=access_start,
                                duration=access_duration,
                            )
                        )
                    # Add access record
                    access_records.append(
                        AccessRecord(
                            target_id=row_to_target_id[gp_index], samples=access_sample
                        )
                    )

    # delete the temporary directory
    shutil.rmtree(temp_dir)

    return AccessResponse(
        **request.model_dump(exclude="target_records"), target_records=access_records
    )

In [ ]:
request = AccessRequest(
    satellites=satellites,
    targets=targets,
    start=mission_start,
    duration=mission_duration,
    propagator=Propagator.SGP4,
    time_step=propagate_time_step,
    payload_ids=["Seasat-SAR"],
)

# display(request.model_dump_json())

access_response = access_orbitpy(request)

# display(access_response.model_dump_json())

access_data = access_response.as_dataframe()

display(access_data)

## Run Data Metrics Calculation with InstruPy

In [ ]:
def get_instantaneous_data_metrics_object(instru_type, time_instant, data_metrics):
    """
    Create an instantaneous data metrics object for the specified sensor type.

    :param instru_type: The type of the instrument (e.g., "SinglePolStripMapSARInstantaneous").
    :paramtype instru_type: str
    :param time_instant: The time at which the data metrics are recorded.
    :paramtype time_instant: AwareDatetime
    :param data_metrics: A dictionary containing data metrics such as NESZ, pixel resolutions, etc.
    :paramtype data_metrics: dict
    :return: An instance of `SinglePolStripMapSARInstantaneous` with the provided metrics.
    :rtype: SinglePolStripMapSARInstantaneous
    :raises ValueError: If the sensor type is not supported.
    """
    if instru_type == "SinglePolStripMapSAR":
        return SinglePolStripMapSARInstantaneous(
            time=time_instant,
            noise_equivalent_sigma_zero=data_metrics["NESZ [dB]"],
            along_track_resolution=data_metrics["ground pixel along-track resolution [m]"],
            cross_track_resolution=data_metrics["ground pixel cross-track resolution [m]"],
            incidence_angle=data_metrics["incidence angle [deg]"],
            swath_width=data_metrics["swath-width [km]"],
            pulse_repetition_frequency=data_metrics["PRF [Hz]"]
        )
    else:
        raise ValueError(
            f"{instru_type} instrument type is not supported in this script. Only 'SinglePolStripMapSAR' instrument type is supported."
        )

def data_metrics_instrupy(request):
    """
    Calculate data metrics using the provided request details and return the results.

    :param request: A `DataMetricsRequest` object containing target records, satellites, satellite records, and other metadata.
    :paramtype request: DataMetricsRequest
    :return: A `DataMetricsResponse` object containing the calculated data metrics for the requested targets and instruments.
    :rtype: DataMetricsResponse
    :raises ValueError: If the sensor type is not supported.
    """

    def get_propagation_record(propagation_records, satellite_id):
        """
        Find the PropagationRecord corresponding to the satellite-id.

        :param propagation_records: List of satellite propagation records.
        :paramtype propagation_records: list[PropagationRecord]
        :param satellite_id: The unique identifier of the satellite.
        :paramtype satellite_id: str
        :return: The matching PropagationRecord object.
        :rtype: PropagationRecord
        :raises RuntimeError: If the propagation record for the requested satellite-id is not found.
        """
        for record in propagation_records:
            if record.satellite_id == satellite_id:
                return record
        raise RuntimeError(
            "Propagation record for requested satellite-id was not found."
        )

    def get_instrument_model(satellites, satellite_id, instrument_id):
        """
        Find the instrument model corresponding to the instrument-id among the list of instruments in the satellite.

        :param satellites: List of satellite objects.
        :paramtype satellites: list[Satellite]
        :param satellite_id: The unique identifier of the satellite.
        :paramtype satellite_id: str
        :param instrument_id: The identifier of the instrument. The identifier of the instrument. The identifier needs to be unique within the list of instrument ids for a given satellite object.
        :paramtype instrument_id: str
        :return: The instrument type and corresponding sensor model.
        :rtype: tuple[str, InstruPy_PassiveOpticalScannerModel]
        :raises ValueError: If the instrument type is unsupported.
        :raises RuntimeError: If the instrument model is not found within the specified satellite.
        """
        for sat in satellites:
            if sat.id == satellite_id:
                for instru in sat.payloads:
                    if instru.id == instrument_id:
                        instru_type = instru.type
                        if instru_type == "SinglePolStripMapSAR":
                            instrupy_fov_geom = None
                            if instru.field_of_view.type == "CircularGeometry":
                                instrupy_fov_geom = {
                                    "shape": "CIRCULAR",
                                    "diameter": instru.field_of_view.diameter,
                                }
                            elif instru.field_of_view.type == "RectangularGeometry":
                                instrupy_fov_geom = {
                                    "shape": "RECTANGULAR",
                                    "angleHeight": instru.field_of_view.angle_height,
                                    "angleWidth": instru.field_of_view.angle_width,
                                }
                            else:
                                raise ValueError(
                                    f"Only Circular and Rectangular geometries are supported and not {instru.field_of_view.type}"
                                )
                            
                            instrupy_scene_fov_geom = None
                            if instru.scene_field_of_view: 
                                if instru.scene_field_of_view.type == "CircularGeometry":
                                    instrupy_scene_fov_geom = {
                                        "shape": "CIRCULAR",
                                        "diameter": instru.scene_field_of_view.diameter,
                                    }
                                elif instru.scene_field_of_view.type == "RectangularGeometry":
                                    instrupy_scene_fov_geom = {
                                        "shape": "RECTANGULAR",
                                        "angleHeight": instru.scene_field_of_view.angle_height,
                                        "angleWidth": instru.scene_field_of_view.angle_width,
                                    }
                                else:
                                    raise ValueError(
                                        f"Only Circular and Rectangular geometries are supported and not {instru.instrupy_scene_fov_geom.type}"
                                    )

                            # Convert orientation in Quaternion to Euler rotations
                            r = Scipy_Rotation.from_quat(list(instru.orientation))
                            (x, y, z) = r.as_euler(
                                "XYZ", degrees=True
                            )  # Conventions 'XYZ' are for intrinsic rotations (used by OrbitPy), while 'xyz' are for extrinsic rotations.
                            if (abs(x) < 1e-6 or abs(z) < 1e-6): # only side-looking geometries are supported for SAR. 
                                instrupy_orientation = {"referenceFrame": "SC_BODY_FIXED", "convention": "SIDE_LOOK", "sideLookAngle":y}
                            else:
                                raise RuntimeError("Only side-looking orientation of instrument supported for the synthetic aperture radar imaging. Rotations about the x and z axis are checked to be zero.")  

                            # Get the InstruPy Antenna specifications 
                            instrupy_antenna = None
                            if instru.antenna.shape.type == "RectangularAntennaShape":
                                instrupy_antenna = {"shape": "RECTANGULAR", 
                                                    "apertureExcitationProfile": instru.antenna.aperture_excitation_profile,
                                                    "diameter": None,
                                                    "height": instru.antenna.shape.height, 
                                                    "width":instru.antenna.shape.width,
                                                    "apertureEfficiency": instru.antenna.aperture_efficiency,
                                                    "radiationEfficiency": instru.antenna.radiation_efficiency,
                                                    "phyTemp": instru.antenna.physical_temp
                                                    }
                            elif instru.antenna.shape == "CircularAntennaShape":
                                instrupy_antenna = {"shape": "CIRCULAR", 
                                                    "apertureExcitationProfile": instru.antenna.aperture_excitation_profile,
                                                    "diameter": instru.antenna.shape.diameter,
                                                    "height": None, 
                                                    "width":None,
                                                    "apertureEfficiency": instru.antenna.aperture_efficiency,
                                                    "radiationEfficiency": instru.antenna.radiation_efficiency,
                                                    "phyTemp": instru.antenna.physical_temp
                                                    }

                            else:
                                raise ValueError(
                                    f"Only Circular and Rectangular antenna shapes are supported and not {instru.antenna.shape.type}"
                                )
                             
                            instrupy_sensor = InstruPy_SyntheticApertureRadarModel.from_dict(
                                {
                                    "@type": "Synthetic Aperture Radar",
                                    "mass": instru.mass, 
                                    "volume": instru.volume, 
                                    "power": instru.power,
                                    "orientation": instrupy_orientation,
                                    "fieldOfViewGeometry": instrupy_fov_geom, 
                                    "sceneFieldOfViewGeometry": instrupy_scene_fov_geom,                                    
                                    "maneuver": None,
                                    "pointingOption": None,
                                    "dataRate": instru.data_rate,
                                    "bitsPerPixel": instru.bits_per_pixel,
                                    "pulseWidth": instru.pulse_width,
                                    "antenna": instrupy_antenna,
                                    "operatingFrequency": instru.operating_frequency,
                                    "peakTransmitPower": instru.peak_transmit_power,
                                    "chirpBandwidth": instru.chirp_bandwidth,
                                    "minimumPRF": instru.minimum_prf,
                                    "maximumPRF": instru.maximum_prf,
                                    "radarLoss": instru.radar_loss,
                                    "atmosLoss": instru.atmos_loss,
                                    "sceneNoiseTemp": instru.scene_noise_temp,
                                    "systemNoiseFigure": instru.system_noise_figure,
                                    "polType": "SINGLE",
                                    "dualPolPulseConfig": None,
                                    "dualPolPulseSep": None,
                                    "swathType": None,
                                    "scanTechnique": "Stripmap",
                                    "fixedSwathSize": None,
                                    "numSubSwaths": None,
                                    "@id": instru.id,
                                }
                            )
                            print(instrupy_sensor.get_field_of_view())
                            return instru_type, instrupy_sensor                        
                        else:
                            raise ValueError(
                                f"{instru_type} instrument type is not supported by this script. Only 'SinglePolStripMapSAR' instrument type is supported."
                            )
        raise RuntimeError(
            "Instrument model for the requested instrument-id (within the specified satellite_id) was not found."
        )

    def propagation_samples_within_time_range(
        propagation_samples, start_time, stop_time
    ):
        """
        Return propagation records within the specified time range.

        :param propagation_samples: List of propagation samples containing satellite states.
        :paramtype propagation_samples: list[PropagationSample]
        :param start_time: The start time of the range.
        :paramtype start_time: AwareDatetime
        :param stop_time: The stop time of the range.
        :paramtype stop_time: AwareDatetime
        :return: A list of filtered propagation records within the time range.
        :rtype: list[PropagationSample]
        """
        return [
            prop_record
            for prop_record in propagation_samples
            if start_time <= prop_record.time <= stop_time
        ]

    def get_target_position(target_id, all_targets):
        """
        Get the position of the target by its ID.

        :param target_id: The unique identifier of the target.
        :paramtype target_id: str
        :param all_targets: List of all targets.
        :paramtype all_targets: list[TargetPoint]
        :return: The position of the target as a tuple of (longitude, latitude).
        :rtype: Position
        :raises RuntimeError: If the target is not found.
        """
        for target in all_targets:
            if target.id == target_id:
                return target.position
        raise RuntimeError(f"Target with id {target_id} was not found.")

    # Main processing
    access_target_records = request.target_records
    dm_req_start = request.start
    dm_req_duration = request.duration

    dm_record = (
        []
    )  # Aggregation of data-metrics across multiple targets and multiple overpasses for each target.
    for access_record in access_target_records:
        target_position = get_target_position(access_record.target_id, request.targets)

        dm_sample = (
            []
        )  # Aggregation of data-metrics over multiple overpasses for the target
        for access_sample in access_record.samples:
            _sat_id = access_sample.satellite_id
            _instru_id = access_sample.instrument_id
            _start = max(dm_req_start, access_sample.start)
            _stop = min(
                dm_req_start + dm_req_duration,
                access_sample.start + access_sample.duration,
            )

            propagation_record = get_propagation_record(
                request.satellite_records, _sat_id
            )
            pr = propagation_samples_within_time_range(
                propagation_record.samples, _start, _stop
            )

            instru_type, instrupy_sensor = get_instrument_model(
                request.satellites, _sat_id, _instru_id
            )

            dm_inst = (
                []
            )  # Aggregation of data-metrics over a single overpass (=coverage sample) for the target
            if pr:
                for _pr in pr:
                    time_utc = AstroPy_Time(
                        _pr.time.astimezone(timezone.utc), scale="utc"
                    )
                    time_ut1 = time_utc.ut1
                    jd_ut1 = time_ut1.jd

                    SpacecraftOrbitState = {
                        "time [JDUT1]": jd_ut1,
                        "x [km]": _pr.position[0] * 1e-3,
                        "y [km]": _pr.position[1] * 1e-3,
                        "z [km]": _pr.position[2] * 1e-3,
                        "vx [km/s]": _pr.velocity[0] * 1e-3,
                        "vy [km/s]": _pr.velocity[1] * 1e-3,
                        "vz [km/s]": _pr.velocity[2] * 1e-3,
                    }
                    TargetCoords = {
                        "lat [deg]": target_position[1],
                        "lon [deg]": target_position[0],
                    }
                    data_metrics = instrupy_sensor.calc_data_metrics(
                        SpacecraftOrbitState, TargetCoords
                    )

                    _dm_inst = get_instantaneous_data_metrics_object(
                        instru_type, _pr.time, data_metrics
                    )
                    dm_inst.append(_dm_inst)

            _dm_sample = DataMetricsSample(
                **access_sample.model_dump(exclude="instantaneous_metrics"),
                instantaneous_metrics=dm_inst,
            )
            dm_sample.append(_dm_sample)

        _dm_record = DataMetricsRecord(
            **access_record.model_dump(exclude="samples"), samples=dm_sample
        )
        dm_record.append(_dm_record)

    data_metrics_response = DataMetricsResponse(
        **request.model_dump(exclude="target_records"), target_records=dm_record
    )
    return data_metrics_response

In [ ]:
request = DataMetricsRequest(
    start=mission_start,
    duration=mission_duration,
    **access_response.model_dump(exclude=["start", "duration", "propagator"]),
    **tatc_propagation_response.model_dump(
        exclude=["start", "duration", "satellites", "time_step"]
    ),
)

print("Data metrics request")
display(request.model_dump_json())

data_metrics_response = data_metrics_instrupy(request)

print("Data metrics response")
display(data_metrics_response.model_dump_json())

In [ ]:
display(data_metrics_response.target_records[4].model_dump_json())

In [ ]:
display(data_metrics_response.target_records[33].model_dump_json())